# Walkthrough

A walkthrough of the entire experiment for an eligible complex. Expects data under `/src/data/`:

- `/src/data/diffdock`: contains candidate poses per complex.
- `/src/data/posebusters`: contains protein files.

In [1]:
import os
import pandas as pd

import utils, filter, prepare, encode
import importlib
importlib.reload(utils)
importlib.reload(filter)
importlib.reload(prepare)
importlib.reload(encode)

<module 'encode' from '/Users/mattgc/code/thesis-custom/research/sapt-preproc/src/encode.py'>

In [ ]:
SRC = os.getcwd()
DATA = os.path.join(SRC, 'data')
DIFFDOCK = os.path.join(DATA, "diffdock")
POSEBUSTERS = os.path.join(DATA, "posebusters")
OUT = os.path.join(SRC, 'out')
RESULTS = os.path.join(OUT, 'results')

# Pre-Processing

In [ ]:
# WARNING: takes several hours
filter.run(name="filter_v1_1_mm")

In [33]:
df = pd.read_csv("out/filter_v1_1_mm_unsize.csv")
df.head()

,name,status,heavy_atoms,charge,electrons,poses,excluded,near_native,rejection
0,5SAK_ZRY,eligible,300.0,-6.0,2282.0,36,4.0,7,NaN
1,5SB2_1K2,eligible,297.0,-1.0,2306.0,40,0.0,40,NaN
2,5SD5_HWI,rejected,NaN,NaN,NaN,40,NaN,0,ligand carrying a group ionised at pH 7.4
3,6M2B_EZO,rejected,NaN,NaN,NaN,40,NaN,0,ligand carrying a group ionised at pH 7.4
4,6M73_FNR,rejected,NaN,NaN,NaN,40,NaN,0,ligand carrying a group ionised at pH 7.4


In [36]:
df["difficulty"] = df["near_native"] / df["poses"]
df = df[
    (df["status"] == "eligible") 
        & (df["difficulty"] < 0.5) 
        # & (-2 < df["charge"])
]
df = df.sort_values(by=["electrons"])
df

,name,status,heavy_atoms,charge,electrons,poses,excluded,near_native,rejection,difficulty
59,7LOE_Y84,eligible,147.0,0.0,1144.0,40,0.0,1,NaN,0.025000
46,7F5D_EUO,eligible,163.0,-3.0,1226.0,40,0.0,8,NaN,0.200000
149,7W05_GMP,eligible,183.0,-1.0,1364.0,39,1.0,5,NaN,0.128205
84,7OEO_V9Z,eligible,210.0,-1.0,1594.0,40,0.0,7,NaN,0.175000
88,7OSO_0V1,eligible,228.0,-1.0,1720.0,37,3.0,8,NaN,0.216216
161,7XFA_D9J,eligible,231.0,1.0,1760.0,27,12.0,4,NaN,0.148148
167,7Z2O_IAJ,eligible,251.0,0.0,1870.0,40,0.0,3,NaN,0.075000
102,7Q2B_M6H,eligible,268.0,-3.0,2038.0,37,3.0,1,NaN,0.027027
191,8DSC_NCA,eligible,274.0,1.0,2088.0,40,0.0,15,NaN,0.375000
0,5SAK_ZRY,eligible,300.0,-6.0,2282.0,36,4.0,7,NaN,0.194444


In [ ]:
df.to_csv("out/populated_v1_1_mm.csv")

## Preparation

Before the protein (monomer A) and poses (monomer Bs) are encoded, the complex must be prepared. Preparation loads the proteins and poses, verifies they are in scope, cleans them, fixes them, protonates both sides, minimises the poses, reverifies, then finally calculates the net charge and verifies that the number of electrons is even.